<a href="https://colab.research.google.com/github/Gaayu25/RAG_PROJECT/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    sentence-transformers

In [ ]:
import langchain
import faiss
import pypdf

print("LangChain:", langchain.__version__)
print("FAISS: OK")
print("PyPDF: OK")
print("All libraries are ready!")

LangChain: 1.3.18
FAISS: OK
PyPDF: OK
All libraries are ready!


In [ ]:
import os
import getpass

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

print("Libraries imported successfully!")

/tmp/ipykernel_1144/1680462278.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries imported successfully!


In [ ]:
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass(
    "Enter your Google Gemini API Key: "
)

Enter your Google Gemini API Key: ··········


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded file:", pdf_file)

Saving medical_reference_rag_sample.pdf to medical_reference_rag_sample.pdf
Uploaded file: medical_reference_rag_sample.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print("PDF loaded successfully!")
print("Number of pages:", len(documents))

PDF loaded successfully!
Number of pages: 2


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Text splitting completed!")
print("Number of chunks:", len(chunks))

Text splitting completed!
Number of chunks: 5


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")

Retriever created successfully!


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini connected successfully!")

Gemini connected successfully!


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a medical information assistant.

Answer the question using ONLY the information
provided in the context.

Do not diagnose a person.
Do not prescribe medicines.
Do not recommend changing or stopping treatment.

If the answer is not available in the context, say:

"I could not find this information in the uploaded reference."

Context:
{context}

Question:
{question}

Answer:
""")

print("RAG prompt created successfully!")

RAG prompt created successfully!


In [ ]:
def ask_question(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)

    print("\n================================")
    print("MEDICAL INFORMATION")
    print("================================")

    print(response.content)

    print("\n================================")
    print("REFERENCE SOURCES")
    print("================================")

    for i, doc in enumerate(retrieved_docs, 1):

        page = doc.metadata.get("page", "Unknown")

        if isinstance(page, int):
            page += 1

        print(f"Source {i}: Page {page}")

In [ ]:
question = input("Enter your question: ")

ask_question(question)